# 02 — Analyse exploratoire (EDA)

**Données** : `data/processed/train.parquet` uniquement (80 000 clients, D34). Le jeu de test n'est pas lu.

**Structure** (sujet de l'atelier) :
1. Analyse univariée : chaque variable seule.
2. Analyse bivariée : chaque variable × churn, puis relations entre variables explicatives.
3. Analyse multivariée et segmentation.

**Lecture des taux** : l'échantillon est équilibré à ~50 % de churners. Les taux affichés servent à **comparer des groupes entre eux**, pas à estimer le taux réel d'un opérateur (voir D3). Les intervalles de confiance à 95 % sont ceux de Wilson.

Les calculs sont dans `churn.eda`, les graphiques dans `churn.charts` ; chaque graphique est enregistré en PNG dans `reports/figures/` (préfixe `02_`).

In [1]:
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display

from churn import eda
from churn.charts import (boxplots_grid, category_bars, correlation_heatmap, histograms_grid,
                          publish, rate_chart, rate_heatmap, target_bar)
from churn.data.split import load_train
from churn.logging_setup import setup_logging

setup_logging()
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 80)

train = load_train()
TARGET = "churn"
OVERALL = 100 * train[TARGET].mean()
CATEGORICALS = [c for c in train.select_dtypes("category").columns if c != "ethnic"]
print(train.shape, f"taux de churn moyen : {OVERALL:.2f} %")

(80000, 104) taux de churn moyen : 49.56 %


# PARTIE 1 — Analyse univariée

## 1.1 Variable cible

In [2]:
counts = train[TARGET].value_counts().sort_index()
display(counts.to_frame("clients").assign(part=lambda d: (100 * d.clients / d.clients.sum()).round(2)))
publish(target_bar(counts, "La cible est équilibrée par construction : 49,6 % de churners"), "02_target")

,clients,part
churn,,
0,40350,50.44
1,39650,49.56


**Conclusion** : 39 650 churners sur 80 000 (49,6 %). Cette répartition vient d'un **échantillonnage volontaire** : chez un opérateur, le churn mensuel est de l'ordre de quelques pourcents (hypothèse de travail : 2 % par mois, D3). Conséquences : pas de rééchantillonnage (D2), l'accuracy n'a pas de sens métier, et les probabilités du modèle devront être corrigées vers le taux réel avant tout calcul métier.

## 1.2 Variables numériques, par bloc

### Statistiques descriptives

Moyenne, médiane, écart-type, min, max, asymétrie (*skewness*), aplatissement (*kurtosis*, en excès : 0 pour une loi normale), part de zéros et de manquants.

In [3]:
block_stats = {}
for block, cols in eda.VARIABLE_BLOCKS.items():
    stats = eda.describe_numeric(train, cols)
    block_stats[block] = stats
    print(f"\n=== {block} ({len(cols)} variables)")
    display(stats)


=== Revenu / facture (10 variables)


,moyenne,mediane,ecart_type,min,max,asymetrie,aplatissement,pct_zero,pct_manquant
rev_Mean,58.77,48.27,46.81,0.00,3843.26,10.30,570.41,0.01,0.37
totmrc_Mean,46.25,44.99,23.63,0.00,409.99,1.66,9.41,0.33,0.39
ovrrev_Mean,13.56,1.00,30.74,0.00,1102.40,6.46,88.64,42.51,0.37
vceovr_Mean,13.29,0.70,30.24,0.00,896.09,6.21,76.98,47.06,0.37
datovr_Mean,0.26,0.00,3.30,0.00,423.54,59.34,5531.03,85.95,0.37
totrev,1031.49,804.58,850.60,3.65,27321.50,3.85,36.17,0.00,0.00
adjrev,959.72,738.28,837.71,2.40,27071.30,3.86,36.81,0.00,0.00
avgrev,57.98,49.92,36.28,0.48,924.27,3.24,26.98,0.00,0.00
avg3rev,59.26,48.00,46.89,1.00,1593.00,4.61,53.46,0.00,0.00
avg6rev,58.74,50.00,40.81,1.00,855.00,3.49,26.95,0.00,2.84



=== Usage (41 variables)


,moyenne,mediane,ecart_type,min,max,asymetrie,aplatissement,pct_zero,pct_manquant
mou_Mean,514.35,356.25,526.12,0.0,12206.75,2.34,11.40,1.61,0.37
ovrmou_Mean,41.04,2.75,98.05,0.0,4320.75,7.79,140.83,42.57,0.37
plcd_vce_Mean,145.06,100.33,158.94,0.0,2289.00,2.68,13.24,8.26,0.00
plcd_dat_Mean,0.89,0.00,9.52,0.0,733.67,30.72,1431.70,85.00,0.00
recv_vce_Mean,55.20,26.67,87.74,0.0,3369.33,5.89,84.72,11.74,0.00
recv_sms_Mean,0.05,0.00,2.33,0.0,517.33,159.36,32296.96,99.12,0.00
comp_vce_Mean,109.09,75.67,119.16,0.0,1894.33,2.66,13.27,8.51,0.00
comp_dat_Mean,0.80,0.00,8.53,0.0,559.33,29.08,1240.95,86.57,0.00
mou_cvce_Mean,228.06,146.51,265.52,0.0,4514.45,2.61,11.94,8.51,0.00
mou_cdat_Mean,1.83,0.00,22.69,0.0,3032.05,50.93,4837.51,86.57,0.00



=== Tendance (2 variables)


...

,moyenne,mediane,ecart_type,min,max,asymetrie,aplatissement,pct_zero,pct_manquant
eqpdays,392.32,342.00,256.86,0.00,1823.00,1.05,1.45,0.03,0.14
hnd_price,101.82,99.99,61.01,9.99,499.99,0.48,0.93,0.00,0.87
phones,1.79,1.00,1.31,1.00,28.00,3.27,20.42,0.00,0.00
models,1.55,1.00,0.90,1.00,16.00,2.45,10.16,0.00,0.00



=== Compte (3 variables)


,moyenne,mediane,ecart_type,min,max,asymetrie,aplatissement,pct_zero,pct_manquant
months,18.83,16.0,9.66,6,61,1.06,0.90,0.00,0.0
uniqsubs,1.55,1.0,0.88,1,18,2.54,11.66,0.00,0.0
actvsubs,1.36,1.0,0.63,0,11,2.19,7.46,0.07,0.0



=== Socio-démographie (7 variables)

,moyenne,mediane,ecart_type,min,max,asymetrie,aplatissement,pct_zero,pct_manquant
income,5.78,6.0,2.18,1.0,9.0,-0.39,-0.45,0.00,25.46
adults,2.53,2.0,1.45,1.0,6.0,0.84,-0.17,0.00,23.06
lor,6.16,5.0,4.73,0.0,15.0,0.64,-0.85,2.19,30.21
numbcars,1.57,1.0,0.63,1.0,3.0,0.64,-0.55,0.00,49.39
truck,0.19,0.0,0.39,0.0,1.0,1.58,0.49,79.54,1.76
rv,0.08,0.0,0.28,0.0,1.0,3.02,7.09,90.05,1.76
forgntvl,0.06,0.0,0.23,0.0,1.0,3.81,12.52,92.61,1.76


In [4]:
all_stats = pd.concat(block_stats, names=["bloc", "variable"])
flags = eda.shape_flags(all_stats)
summary = pd.DataFrame({
    "variables": all_stats.groupby(level="bloc", sort=False).size(),
    "tres_asymetriques (|skew| > 2)": flags.groupby(level="bloc", sort=False)["tres_asymetrique"].sum(),
    "concentrees_sur_zero (> 50 %)": flags.groupby(level="bloc", sort=False)["concentree_sur_zero"].sum(),
})
summary.loc["Total"] = summary.sum()
summary

,variables,tres_asymetriques (|skew| > 2),concentrees_sur_zero (> 50 %)
bloc,,,
Revenu / facture,10,9,1
Usage,41,41,11
Tendance,2,2,0
Qualité réseau,7,7,3
Service client,3,3,3
Terminal,4,2,0
Compte,3,2,0
Socio-démographie,7,2,3
Total,77,68,21


**Conclusion** : la quasi-totalité des variables d'usage, de revenu, de qualité réseau et de service client est **très asymétrique** : une majorité de clients aux valeurs modestes et une longue queue de gros consommateurs. Seules les variables de terminal, de compte et socio-démographiques ont des distributions proches de la symétrie. Pour un modèle linéaire, ces variables devront être transformées (`log1p`) dans le pipeline ; les arbres n'en ont pas besoin.

### Histogrammes des principales variables

Échelle logarithmique en abscisse quand |asymétrie| > 2 ; la part de zéros, qu'une échelle log ne peut pas montrer, est indiquée dans le titre du panneau.

In [5]:
publish(histograms_grid(train, eda.KEY_VARIABLES["Revenu / facture"],
                        "Revenu : une facture médiane de 48 $ par mois, avec une longue queue"),
        "02_hist_revenu")

**Conclusion** : `rev_Mean` a une médiane de 48 $ pour une moyenne de 59 $ (asymétrie 10,3). `ovrrev_Mean` (dépassements) est nul pour 42,5 % des clients : la majorité reste dans son forfait. `totmrc_Mean` (abonnement mensuel) est concentré sur quelques paliers tarifaires.

In [6]:
publish(histograms_grid(train, eda.KEY_VARIABLES["Usage"],
                        "Usage : distributions log-normales, l'itinérance concerne une minorité"),
        "02_hist_usage")

**Conclusion** : les minutes (`mou_Mean`, médiane 356 min/mois) et les appels suivent des distributions à peu près log-normales. `roam_Mean` est nul pour 68,6 % des clients (asymétrie 170) : l'itinérance est un comportement rare.

In [7]:
publish(histograms_grid(train, eda.KEY_VARIABLES["Tendance"],
                        "Tendance : l'usage baisse légèrement en médiane (−6,5 min)"),
        "02_hist_tendance")

**Conclusion** : `change_mou` et `change_rev` sont centrées près de zéro (médianes −6,5 min et −0,32 $) avec des queues extrêmes des deux côtés ; ce sont les seules variables signées, d'où l'échelle linéaire.

In [8]:
publish(histograms_grid(train, eda.KEY_VARIABLES["Qualité réseau"],
                        "Qualité réseau : quelques appels échoués par mois pour la plupart des clients"),
        "02_hist_reseau")
publish(histograms_grid(train, eda.KEY_VARIABLES["Service client"],
                        "Service client : 56 % des clients n'ont jamais appelé"),
        "02_hist_service")

**Conclusion** : les appels coupés (`drop_vce_Mean`, médiane 3 par mois) et bloqués sont fréquents mais en petit nombre. `custcare_Mean` est nul pour 55,7 % des clients : la variable sera plus utile comme indicateur « a appelé le service client » que comme valeur continue.

In [9]:
publish(histograms_grid(train, eda.KEY_VARIABLES["Terminal"],
                        "Terminal : des pics d'âge vers 180 et 310 jours, des prix par paliers"),
        "02_hist_terminal")
publish(histograms_grid(train, eda.KEY_VARIABLES["Compte"],
                        "Compte : la plupart des clients ont 1 ligne et moins de 2 ans d'ancienneté"),
        "02_hist_compte")

**Conclusion** : `hnd_price` ne prend que 17 valeurs (paliers de prix catalogue) : c'est presque une catégorielle ordonnée. `eqpdays` a une médiane de 342 jours. `months` a une médiane de 16 mois et varie de 6 à 61. `uniqsubs` et `actvsubs` valent 1 pour la majorité (clients mono-ligne).

### Boxplots

In [10]:
key_vars = [c for cols in eda.KEY_VARIABLES.values() for c in cols]
publish(boxplots_grid(train, key_vars, "Boxplots : des moustaches courtes et des queues très longues"),
        "02_box_univarie", width=1100)

**Conclusion** : pour les variables d'usage et de revenu, la boîte (Q1–Q3) est écrasée près de zéro par rapport à l'étendue totale : c'est la signature des queues lourdes vues à l'audit (D29). Les points au-delà des moustaches ne sont pas tracés (ils représentent 5 à 26 % des clients selon la variable).

## 1.3 Variables catégorielles

`crclscod` (53 modalités) est regroupé par sa lettre initiale (classe de crédit principale), les lettres de moins de 1 % des clients étant réunies dans « Autres ». `ethnic` n'est pas analysée (exclue du projet, D4).

In [11]:
cat_view = train[CATEGORICALS].copy()
cat_view["crclscod"] = eda.credit_class(train["crclscod"])
freqs = {c: eda.category_frequencies(cat_view[c]) for c in CATEGORICALS}
dominance = pd.DataFrame({
    "modalites": {c: len(f) for c, f in freqs.items()},
    "modalite_principale": {c: f.index[0] for c, f in freqs.items()},
    "part_principale": {c: f["pct"].iloc[0] for c, f in freqs.items()},
    "part_manquant": {c: f["pct"].get("Manquant", 0.0) for c, f in freqs.items()},
}).sort_values("part_principale", ascending=False)
dominance

,modalites,modalite_principale,part_principale,part_manquant
kid0_2,3,Unknown,94.23,1.76
kid3_5,3,Unknown,93.56,1.76
kid6_10,3,Unknown,90.13,1.76
kid11_15,3,Unknown,89.40,1.76
kid16_17,3,Unknown,88.22,1.76
asl_flag,2,N,86.02,0.00
refurb_new,3,N,85.53,0.00
infobase,3,M,77.66,22.12
hnd_webcap,4,WCMB,75.70,10.20
dualband,5,Y,72.28,0.00


In [12]:
publish(category_bars({c: freqs[c] for c in ["crclscod", "asl_flag", "refurb_new", "dualband",
                                               "hnd_webcap", "new_cell", "prizm_social_one",
                                               "creditcd", "marital"]},
                      "Compte et terminal : des modalités très dominantes"), "02_cat_compte", width=1100)

In [13]:
publish(category_bars({c: freqs[c] for c in ["ownrent", "dwlltype", "dwllsize", "HHstatin",
                                               "infobase", "kid0_2", "kid6_10", "kid16_17"]},
                      "Socio-démographie : beaucoup de manquants et de « Unknown »"), "02_cat_socio", width=1100)

**Conclusion** : les `kid*` sont dominées à 88–94 % par « Unknown » (aucun enfant renseigné), `asl_flag` à 86 % par N, `refurb_new` à 85,5 % par N (terminal neuf), `infobase` à 78 % par M. `new_cell` vaut « Unknown » pour 67 % des clients : son information est très limitée. La classe de crédit A regroupe 54 % des clients.

In [14]:
publish(category_bars({"area": freqs["area"]}, "Région : aucune zone ne dépasse 11 % des clients",
                      cols=1, max_levels=20), "02_cat_area", width=900)

**Conclusion** : les 19 régions sont assez équilibrées (de 2,4 % à 11 % des clients, New York en tête) ; 30 clients n'ont pas de région.

## Conclusion de la partie 1

- **Très asymétriques** : 68 variables numériques sur 77 (|asymétrie| > 2), en particulier `roam_Mean` (170), `change_rev` (86), `custcare_Mean` (35), `rev_Mean` (10). À transformer (`log1p`) pour les modèles linéaires.
- **Concentrées sur zéro** (> 50 % de zéros) : 21 variables, dont toutes les variables *data* (`*_dat_Mean`), `roam_Mean`, `threeway_Mean`, `callfwdv_Mean`, les 3 variables de service client, `truck`, `rv`, `forgntvl`. Candidates à un indicateur binaire « > 0 ».
- **Dominées par une modalité** : `kid*` (88–94 % « Unknown »), `asl_flag` (86 % N), `refurb_new` (85,5 % N), `infobase` (78 % M), `new_cell` (67 % Unknown).
- **Quasi catégorielles** : `hnd_price` (17 paliers), `months` (entier de 6 à 61).

# PARTIE 2 — Analyse bivariée (variable × churn)

## 2.1 Numériques : distributions churners vs non-churners

In [15]:
medians = train.groupby(TARGET)[key_vars].median().T
medians.columns = ["mediane_non_churners", "mediane_churners"]
medians["ecart_relatif_%"] = (100 * (medians.mediane_churners / medians.mediane_non_churners - 1)).round(1)
medians

,mediane_non_churners,mediane_churners,ecart_relatif_%
rev_Mean,48.935000,47.568750,-2.8
totmrc_Mean,44.990000,44.022500,-2.2
ovrrev_Mean,0.800000,1.225000,53.1
totrev,783.680000,825.365000,5.3
mou_Mean,381.250000,329.750000,-13.5
ovrmou_Mean,2.250000,3.500000,55.6
totcalls,1774.000000,1868.500000,5.3
roam_Mean,0.000000,0.000000,NaN
change_mou,-3.000000,-10.000000,233.3
change_rev,-0.312500,-0.337500,8.0


In [16]:
publish(boxplots_grid(train, key_vars, "Les churners utilisent moins leur forfait et ont des terminaux plus vieux",
                      by=TARGET), "02_box_par_churn", width=1100)

**Conclusion** : les distributions des deux classes se recouvrent largement ; aucune variable ne sépare à elle seule churners et non-churners. Les écarts de médiane les plus nets :

- terminal plus vieux chez les churners (`eqpdays` 366 j contre 310, +18 %) et moins cher (`hnd_price` 99,99 $ contre 129,99 $) ;
- usage plus faible (`mou_Mean` 330 min contre 381, −13,5 %) et en baisse plus marquée (`change_mou` −10 contre −3) ;
- moins de contacts avec le service client (Q3 de `custcare_Mean` plus bas).

Revenu, qualité réseau et nombre de lignes ne diffèrent presque pas.

## 2.2 Numériques : taux de churn par décile

Classement de toutes les variables numériques par **amplitude** du taux de churn entre leurs déciles (taux max − taux min). `rho` est la corrélation de Spearman avec le churn.

In [17]:
all_numeric = [c for cols in eda.VARIABLE_BLOCKS.values() for c in cols]
contrast = eda.decile_contrast(train, all_numeric)
contrast.head(15)

,n_classes,taux_min,taux_max,amplitude_pts,rho
variable,,,,,
months,10,32.99,61.71,28.7,0.046
eqpdays,10,37.04,60.90,23.9,0.129
hnd_price,8,36.80,57.64,20.8,-0.104
mou_Mean,10,44.36,58.85,14.5,-0.068
totmrc_Mean,10,43.48,56.91,13.4,-0.073
mou_opkv_Mean,10,44.79,57.64,12.8,-0.072
complete_Mean,10,44.52,57.25,12.7,-0.061
avg3mou,10,44.94,57.58,12.6,-0.058
peak_vce_Mean,10,44.85,57.49,12.6,-0.057


**Conclusion** : trois variables dominent nettement : `months` (28,7 points d'écart entre déciles), `eqpdays` (23,9) et `hnd_price` (20,8). Viennent ensuite une douzaine de variables d'usage à 12-14 points, qui portent **la même information** (elles sont très corrélées, voir 2.5).

Point de méthode : `months` a la plus forte amplitude mais une corrélation de Spearman quasi nulle (ρ = 0,046). La relation est **non monotone** (pic à 11-12 mois, voir 2.3) : une corrélation, même de rang, ne la voit pas. D'où l'intérêt d'analyser par classes, et de modèles capables de capter des effets non linéaires.

In [18]:
publish(rate_chart(eda.churn_rate_table(train, eda.decile_bins(train["mou_Mean"])),
                   "Moins un client consomme de minutes, plus il churne", OVERALL,
                   "décile de mou_Mean (minutes / mois)"), "02_rate_mou")

**Conclusion** : le taux de churn baisse régulièrement avec l'usage, de 51,6 % (2e décile, 49 à 116 min) à 44,4 % (dernier décile, plus de 1 178 min). Le 1er décile (moins de 49 min, dont les clients à 0 min) monte à 58,9 %, et les clients sans aucune donnée d'usage à 68,9 % (risque de fuite « déjà parti », D32).

In [19]:
publish(rate_chart(eda.churn_rate_table(train, eda.decile_bins(train["totmrc_Mean"])),
                   "Les petits forfaits churnent davantage que les gros", OVERALL,
                   "décile de totmrc_Mean (abonnement mensuel, $)"), "02_rate_totmrc")

**Conclusion** : les petits forfaits (moins de 30 $ par mois) churnent à 53-57 %, les forfaits de 60 $ et plus à 43-44 %. Un forfait élevé va souvent avec un engagement plus fort, ou avec un client mieux équipé et plus attaché au service.

## 2.3 Analyses prioritaires

### `eqpdays` : l'hypothèse d'une fin d'engagement vers 12 mois

In [20]:
edges = list(range(0, 931, 30)) + [np.inf]
labels = [f"{a}-{a + 29}" for a in range(0, 930, 30)] + [">= 930"]
eqp_table = eda.churn_rate_table(train, eda.fixed_bins(train["eqpdays"], edges, labels))
publish(rate_chart(eqp_table, "Le churn saute de 37 % à 60 % quand le terminal passe 300 jours",
                   OVERALL, "âge du terminal (jours, tranches de 30)"), "02_rate_eqpdays", width=1100)

In [21]:
zoom = train[train["eqpdays"].between(250, 399)]
zoom_bins = eda.fixed_bins(zoom["eqpdays"], list(range(250, 401, 5)), [f"{a}-{a + 4}" for a in range(250, 400, 5)])
zoom_table = eda.churn_rate_table(zoom, zoom_bins)
display(zoom_table.loc["290-294":"325-329", ["n", "taux", "ic_bas", "ic_haut"]])
publish(rate_chart(zoom_table, "Zoom : la rupture se situe entre 300 et 309 jours", OVERALL,
                   "âge du terminal (jours, tranches de 5)"), "02_rate_eqpdays_zoom", width=1100)

,n,taux,ic_bas,ic_haut
eqpdays,,,,
290-294,720,35.83,32.41,39.40
295-299,679,38.88,35.29,42.60
300-304,640,42.03,38.27,45.89
305-309,1062,58.95,55.96,61.87
310-314,1184,60.05,57.23,62.80
315-319,991,60.04,56.96,63.05
320-324,1026,65.69,62.73,68.53
325-329,995,65.63,62.62,68.51


**Conclusion : l'hypothèse est validée, avec une précision.** Le churn reste entre 37 et 44 % jusqu'à 299 jours, puis **saute entre 300-304 jours (42,0 %) et 305-309 jours (59,0 %)**, avant de redescendre vers 52-56 % et de remonter lentement pour les terminaux très anciens (60 % au-delà de 900 jours). La rupture se situe vers **10 mois**, et non 12.

Interprétation (hypothèse) : si le churn est mesuré sur une fenêtre de un à deux mois après l'observation (cas de Cell2Cell), un terminal de 10 mois atteint ses 12 mois d'engagement **pendant** cette fenêtre. On note aussi un pic d'effectif juste après 300 jours (renouvellements groupés). Ce lien est une association, pas une preuve causale : il faudrait la date de fin de contrat pour conclure.

### `months` : ancienneté

In [22]:
m = train["months"].where(train["months"] <= 36, 37)
months_table = eda.churn_rate_table(train, m).rename(index={37: "37+"})
display(months_table.loc[[6, 7, 8, 9, 10, 11, 12, 13, 14], ["n", "taux", "ic_bas", "ic_haut"]])
publish(rate_chart(months_table, "Le churn culmine à 11-12 mois d'ancienneté, puis se stabilise",
                   OVERALL, "ancienneté (mois)"), "02_rate_months", width=1100)

,n,taux,ic_bas,ic_haut
months,,,,
6,1047,38.49,35.59,41.48
7,3985,32.92,31.48,34.40
8,3008,31.15,29.52,32.83
9,3255,28.66,27.14,30.24
10,3485,40.37,38.76,42.01
11,6407,64.82,63.64,65.98
12,4844,61.71,60.33,63.06
13,4490,55.55,54.09,56.99
14,3634,52.75,51.13,54.37


**Conclusion** : le churn est faible de 6 à 10 mois (29 à 40 %), **bondit à 11 mois (64,8 %) et 12 mois (61,7 %)**, puis décroît vers 50 % dès 15 mois. Même signature que `eqpdays` : les clients sous engagement partent peu, le départ se concentre à l'approche de l'échéance. Les 11-12 mois représentent 14,1 % des clients du train. L'effectif est lui-même maximal à 11 mois (6 407 clients), sans doute un effet de la constitution de l'échantillon.

### `change_mou` : évolution de l'usage

In [23]:
change_table = eda.churn_rate_table(train, eda.decile_bins(train["change_mou"]))
display(change_table[["n", "taux", "ic_bas", "ic_haut"]])
publish(rate_chart(change_table, "Une baisse d'usage s'accompagne de plus de churn ; l'absence de mesure encore plus",
                   OVERALL, "décile de change_mou (variation de minutes)"), "02_rate_change_mou", width=1100)

,n,taux,ic_bas,ic_haut
change_mou,,,,
"[-3,875 ; -240.5]",7936,55.42,54.32,56.51
[-240.5 ; -119.8],7937,52.61,51.51,53.71
[-119.8 ; -63],7939,50.28,49.18,51.38
[-63 ; -29],7934,48.69,47.59,49.79
[-29 ; -6.5],7936,49.87,48.77,50.97
[-6.5 ; 8.75],7948,51.89,50.79,52.98
[8.75 ; 40],7890,45.82,44.72,46.92
[40 ; 93.5],7910,45.34,44.24,46.43
[93.5 ; 206.5],7924,45.39,44.30,46.49


**Conclusion** : une forte baisse d'usage (1er décile, plus de 240 minutes perdues) correspond à 55,4 % de churn, contre 45-46 % pour une hausse modérée (8 à 206 minutes). La relation n'est pas strictement monotone (remontée à 51,9 % autour de zéro). Surtout, les 718 clients dont la variation est **manquante** churnent à 76,0 % : c'est le signal le plus fort de cette variable, et un candidat à la fuite « client déjà parti » (D32).

### `custcare_Mean > 0` : contacts avec le service client

In [24]:
custcare_group = pd.Series(np.where(train["custcare_Mean"] > 0, "au moins un appel", "aucun appel"),
                           index=train.index, name="service client")
display(eda.churn_rate_table(train, custcare_group))
custcare_level = pd.cut(train["custcare_Mean"], [-0.1, 0, 1, 3, np.inf],
                        labels=["0", "0-1 / mois", "1-3 / mois", "> 3 / mois"]).rename("custcare_Mean")
publish(rate_chart(eda.churn_rate_table(train, custcare_level),
                   "Les clients qui contactent le service client churnent moins", OVERALL,
                   "appels au service client (moyenne mensuelle)"), "02_rate_custcare")

,n,churners,taux,ic_bas,ic_haut,part
service client,,,,,,
au moins un appel,35455,16609,46.85,46.33,47.37,44.32
aucun appel,44545,23041,51.73,51.26,52.19,55.68


**Conclusion** : les clients qui ont contacté le service client churnent un peu **moins** : 46,9 % contre 51,7 % (4,9 points d'écart), et 44,9 % au-delà de 3 appels par mois. L'écart est **faible** et son interprétation reste une **hypothèse**. Plusieurs explications sont possibles :

- le client qui appelle est un client engagé, qui utilise activement son service ;
- une **offre de rétention** a pu lui être proposée pendant l'appel ;
- l'appel a **résolu un problème** qui aurait pu motiver un départ.

Le dataset ne contient ni le motif ni l'issue des appels : on ne peut pas départager ces explications. Le croisement avec l'usage (3.2) vérifie seulement que l'écart n'est pas un simple effet de volume.

### `hnd_price` : prix du terminal

In [25]:
price_table = eda.churn_rate_table(train, train["hnd_price"].round(2))
display(price_table[price_table["n"] >= 100][["n", "taux", "ic_bas", "ic_haut"]])
publish(rate_chart(price_table, "Les terminaux bon marché sont associés à beaucoup plus de churn",
                   OVERALL, "prix du terminal ($, paliers d'au moins 100 clients)", min_n=100),
        "02_rate_hnd_price")

,n,taux,ic_bas,ic_haut
hnd_price,,,,
9.99,3514,58.37,56.73,59.99
29.99,17399,57.50,56.76,58.23
39.99,524,57.06,52.79,61.23
59.99,6919,51.68,50.51,52.86
79.99,7648,49.19,48.07,50.31
99.99,6202,47.18,45.94,48.42
129.99,10691,46.30,45.36,47.25
149.99,17666,46.70,45.96,47.44
199.99,8174,40.96,39.90,42.03


**Conclusion** : les terminaux les moins chers (9,99 $ et 29,99 $, soit 26 % des clients) sont associés à 57-58 % de churn, contre 41 % à 199,99 $. Un terminal bon marché signale un attachement plus faible (entrée de gamme, souvent offert à l'abonnement). Les paliers de moins de 100 clients sont masqués.

### `area` : région

In [26]:
area_table = eda.churn_rate_table(train, train["area"], sort_by_rate=True)
publish(rate_chart(area_table, "10 points d'écart de churn entre régions", OVERALL, horizontal=True),
        "02_rate_area", width=900)

**Conclusion** : le churn va de 45,9 % (DC/Maryland/Virginia) à 56,1 % (Northwest/Rocky Mountain), soit 10 points d'écart ; la plupart des régions sont entre 46 et 52 %. Effet réel mais secondaire (V de Cramér 0,05), qui peut refléter la couverture réseau ou la concurrence locale : le dataset ne permet pas de trancher.

## 2.4 Catégorielles : taux de churn par modalité

Classement par V de Cramér (χ² de la table modalité × churn). L'écart de taux ne considère que les modalités d'au moins 500 clients.

In [27]:
cat_for_test = cat_view.assign(**{TARGET: train[TARGET]})
cat_contrast = eda.categorical_contrast(cat_for_test, CATEGORICALS)
cat_contrast

,n_modalites,modalite_min,taux_min,modalite_max,taux_max,ecart_pts,cramers_v
variable,,,,,,,
hnd_webcap,4,WCMB,47.04,Manquant,58.94,11.9,0.092
crclscod,7,E,39.65,B,52.31,12.7,0.069
asl_flag,2,Y,41.09,N,50.94,9.8,0.068
dualband,5,T,42.14,N,53.95,11.8,0.056
area,20,DC/MARYLAND/VIRGINIA AREA,45.88,NORTHWEST/ROCKY MOUNTAIN AREA,56.09,10.2,0.050
marital,6,Manquant,46.84,Unknown,51.41,4.6,0.031
infobase,3,M,48.91,Manquant,52.04,3.1,0.031
refurb_new,3,N,48.93,R,53.28,4.4,0.031
dwllsize,16,O,47.85,J,51.59,3.7,0.027


In [28]:
top_cats = ["hnd_webcap", "crclscod", "asl_flag", "dualband", "refurb_new"]
combined = pd.concat({c: eda.churn_rate_table(cat_for_test, cat_for_test[c].astype(object))
                      for c in top_cats})
combined.index = [f"{v} = {m}" for v, m in combined.index]
publish(rate_chart(combined, "Webcap, classe de crédit et type de terminal séparent le mieux les clients",
                   OVERALL, horizontal=True, min_n=100), "02_rate_categorielles", width=1000)

**Conclusion** :

- `hnd_webcap` est la catégorielle la plus liée au churn (V = 0,092) : terminal sans capacité web complète (WC) 56,6 % et information manquante 58,9 %, contre 47,0 % pour WCMB ;
- classe de crédit : de 39,7 % (E) à 52,3 % (B) ;
- `asl_flag = Y` (limite de dépense sur le compte) : 41,1 % contre 50,9 % ;
- `dualband` : N 53,9 % contre T (tri-bande) 42,1 %.

Les variables socio-démographiques ont toutes des effets négligeables (V ≤ 0,031), et `new_cell` n'a aucun lien (V = 0,004).

**Tableaux croisés** (part de churners et de non-churners par modalité, en %) :

In [29]:
for col in ["hnd_webcap", "asl_flag", "dualband"]:
    ct = pd.crosstab(cat_for_test[col].astype(object).fillna("Manquant"), train[TARGET],
                     normalize="index", margins=True).mul(100).round(1)
    ct.columns = ["non-churn %", "churn %"]
    ct["n"] = pd.crosstab(cat_for_test[col].astype(object).fillna("Manquant"), train[TARGET],
                          margins=True)["All"]
    print(f"\n{col}")
    display(ct)


hnd_webcap


,non-churn %,churn %,n
hnd_webcap,,,
Manquant,41.1,58.9,8163
Unknown,62.1,37.9,195
WC,43.3,56.7,11085
WCMB,53.0,47.0,60557
All,50.4,49.6,80000



asl_flag


,non-churn %,churn %,n
asl_flag,,,
N,49.1,50.9,68813
Y,58.9,41.1,11187
All,50.4,49.6,80000



dualband


,non-churn %,churn %,n
dualband,,,
Manquant,100.0,0.0,1
N,46.1,53.9,18537
T,57.9,42.1,3450
Unknown,63.6,36.4,184
Y,51.4,48.6,57828
All,50.4,49.6,80000


## 2.5 Relations entre variables explicatives : corrélation de Spearman

Spearman (sur les rangs) plutôt que Pearson : robuste aux queues lourdes et aux valeurs extrêmes.

In [30]:
corr_vars = ["rev_Mean", "totmrc_Mean", "ovrrev_Mean", "ovrmou_Mean", "totrev", "adjrev",
             "mou_Mean", "avg3mou", "avgmou", "totcalls", "roam_Mean", "change_mou", "change_rev",
             "drop_vce_Mean", "blck_vce_Mean", "unan_vce_Mean", "custcare_Mean", "eqpdays",
             "hnd_price", "phones", "months", "uniqsubs", "actvsubs", "income"]
corr = eda.spearman_matrix(train, corr_vars)
publish(correlation_heatmap(corr, "Revenu, usage et qualité réseau forment un bloc très corrélé"),
        "02_spearman", width=1000)

In [31]:
eda.top_correlated_pairs(corr, threshold=0.7)

,var_1,var_2,rho
0,ovrrev_Mean,ovrmou_Mean,1.00
1,totrev,adjrev,1.00
2,mou_Mean,avg3mou,0.98
3,avg3mou,avgmou,0.89
4,mou_Mean,avgmou,0.87
5,uniqsubs,actvsubs,0.83
6,rev_Mean,totmrc_Mean,0.75
7,avgmou,totcalls,0.74
8,totrev,totcalls,0.74
9,adjrev,totcalls,0.74


**Conclusion** : fortes redondances à traiter en E6 :

- **quasi-doublons** (ρ ≥ 0,98) : `ovrrev_Mean` / `ovrmou_Mean` (1,00), `totrev` / `adjrev` (1,00), `mou_Mean` / `avg3mou` (0,98) ;
- **blocs corrélés** : usage (`mou_Mean`, `avgmou`, `totcalls`, ρ de 0,72 à 0,89), revenu et usage (`rev_Mean` / `mou_Mean` 0,72), `uniqsubs` / `actvsubs` (0,83) ;
- `unan_vce_Mean` (appels sans réponse) est corrélée à l'usage (0,70) : c'est surtout un indicateur de volume, pas de mauvaise qualité réseau.

Les variables les plus liées au churn (`eqpdays`, `months`, `hnd_price`) sont **peu corrélées** au reste : elles apportent une information indépendante.

# PARTIE 3 — Analyse multivariée et segmentation

## 3.1 Interaction ancienneté × âge du terminal

In [32]:
months_bins = eda.fixed_bins(train["months"], [0, 11, 13, 18, 24, 36, np.inf],
                             ["6-10", "11-12", "13-17", "18-23", "24-35", "36+"])
eqp_bins = eda.fixed_bins(train["eqpdays"], [0, 180, 300, 365, 540, 730, np.inf],
                          ["0-179", "180-299", "300-364", "365-539", "540-729", "730+"])
rate, n = eda.two_way_rate(train, months_bins, eqp_bins)
publish(rate_heatmap(rate, n, "Le churn culmine à 11-12 mois d'ancienneté, quel que soit l'âge du terminal",
                     OVERALL, "âge du terminal (jours)", "ancienneté (mois)"), "02_heatmap_months_eqpdays")

**Conclusion** :

- la ligne **11-12 mois** est au-dessus de la moyenne quel que soit l'âge du terminal (57 à 65 %) ;
- les clients récents (6-10 mois) avec un terminal de moins de 300 jours churnent peu (31-35 %), mais ceux dont le terminal a déjà 300 jours ou plus montent à 54 % (terminal acheté avant l'abonnement) ;
- chez les clients anciens (36 mois et plus), un terminal récent va avec peu de churn (31 %) et un terminal de 730 jours et plus avec beaucoup (60 %) : **renouveler le terminal semble fidéliser**.

Les deux variables se combinent : un modèle à arbres captera naturellement ces interactions.

## 3.2 Interaction usage × service client

In [33]:
usage_bins = eda.fixed_bins(train["mou_Mean"], eda.USAGE_EDGES, eda.USAGE_LABELS)
care_bins = pd.cut(train["custcare_Mean"], [-0.1, 0, 2, np.inf],
                   labels=["aucun appel", "0-2 appels / mois", "> 2 appels / mois"])
rate, n = eda.two_way_rate(train, usage_bins, care_bins)
publish(rate_heatmap(rate, n, "À usage égal, contacter le service client reste associé à moins de churn",
                     OVERALL, "appels au service client", "usage (minutes / mois)"), "02_heatmap_usage_custcare")

**Conclusion** : à usage égal, le churn reste un peu plus faible chez les clients qui appellent : usage faible 54,7 % → 52,5 %, moyen 50,1 % → 45,9 %, fort 48,5 % → 43,7 %. L'écart n'est donc **pas seulement** un effet d'usage (les gros utilisateurs appellent plus). Il reste **faible** (2 à 5 points) et ne dit rien de sa cause : engagement du client, offre de rétention faite pendant l'appel ou problème résolu restent des hypothèses concurrentes.

## 3.3 Segmentation métier par règles

Trois axes, avec des seuils fixes énonçables à un métier, choisis à partir de la partie 2 (sur le train) :

| Axe | Seuils | Justification |
|---|---|---|
| Ancienneté | 6-10 / 11-12 / ≥ 13 mois | pic de churn à 11-12 mois (fin d'engagement supposée) |
| Usage | < 150 / 150-699 / ≥ 700 min par mois | proches des quartiles du train |
| Âge du terminal | < 300 / 300-729 / ≥ 730 jours | rupture vers 305 jours |

Les clients dont l'une des trois variables manque (usage ou terminal) ne sont pas segmentés.

In [34]:
segments = eda.business_segments(train)
seg_table = eda.segment_table(train, segments)
print(f"{int(seg_table['n'].sum())} clients segmentés, {len(train) - int(seg_table['n'].sum())} non segmentés (donnée manquante)")
seg_table

79593 clients segmentés, 407 non segmentés (donnée manquante)


n   part   taux  ic_bas  ic_haut  ecart_pts
anciennete usage                     terminal                                                           
11-12 mois Usage faible (< 150 min)  Terminal < 300 j      274   0.34  76.64   71.29    81.26      27.08
                                     Terminal 300-729 j   2156   2.70  66.60   64.59    68.56      17.04
           Usage moyen (150-699 min) Terminal 300-729 j   4527   5.66  64.08   62.67    65.47      14.52
           Usage fort (>= 700 min)   Terminal 300-729 j   1839   2.30  62.10   59.86    64.29      12.54
           Usage moyen (150-699 min) Terminal < 300 j     1221   1.53  61.59   58.83    64.28      12.03
>= 13 mois Usage faible (< 150 min)  Terminal >= 730 j    4413   5.52  58.74   57.28    60.18       9.17
           Usage moyen (150-699 min) Terminal >= 730 j    3615   4.52  57.18   55.56    58.78       7.62
11-12 mois Usage fort (>= 700 min)   Terminal < 300 j     1214   1.52  56.92   54.12    59.68       7.36
>= 13 mois Usage faible (< 150 min)  Terminal 300-729 j   8432  10.54  56.70   55.64    57.75       7.14
6-10 mois  Usage fort (>= 700 min)   Terminal 300-729 j    269   0.34  55.02   49.04    60.85       5.46
           Usage moyen (150-699 min) Terminal 300-729 j    675   0.84  54.07   50.30    57.80       4.51
>= 13 mois Usage moyen (150-699 min) Terminal 300-729 j  15777  19.72  53.28   52.50    54.06       3.72
6-10 mois  Usage faible (< 150 min)  Terminal 300-729 j    276   0.34  51.81   45.93    57.64       2.25
>= 13 mois Usage fort (>= 700 min)   Terminal >= 730 j     580   0.72  51.21   47.14    55.25       1.64
                                     Terminal 300-729 j   5230   6.54  49.90   48.55    51.26       0.34
           Usage faible (< 150 min)  Terminal < 300 j     1855   2.32  45.01   42.76    47.29      -4.55
           Usage fort (>= 700 min)   Terminal < 300 j     6415   8.02  41.95   40.75    43.16      -7.61
           Usage moyen (150-699 min) Terminal < 300 j     7293   9.12  41.37   40.24    42.50      -8.19
6-10 mois  Usage fort (>= 700 min)   Terminal < 300 j     4490   5.61  35.48   34.09    36.89     -14.08
           Usage faible (< 150 min)  Terminal < 300 j     2341   2.93  32.25   30.39    34.17     -17.31
           Usage moyen (150-699 min) Terminal < 300 j     6701   8.38  29.49   28.41    30.59     -20.07

In [35]:
seg_plot = seg_table.copy()
seg_plot.index = [" | ".join(idx) for idx in seg_plot.index]
publish(rate_chart(seg_plot, "Les segments vont de 29 % à 77 % de churn : la fin d'engagement domine",
                   OVERALL, horizontal=True, min_n=200), "02_segments", width=1100)

In [36]:
high = seg_table[seg_table["taux"] >= 55]
low = seg_table[seg_table["taux"] < 45]
for name, part in [("taux >= 55 %", high), ("taux < 45 %", low)]:
    print(f"{name} : {len(part)} segments, {int(part['n'].sum())} clients "
          f"({part['n'].sum() / len(train):.1%}), churn pondéré {(part['n'] * part['taux']).sum() / part['n'].sum():.1f} %")

taux >= 55 % : 10 segments, 27960 clients (34.9%), churn pondéré 59.8 %
taux < 45 % : 5 segments, 27240 clients (34.1%), churn pondéré 36.8 %


**Conclusion** : la segmentation par règles sépare nettement les clients, de **29,5 %** (6-10 mois, usage moyen, terminal récent) à **76,6 %** (11-12 mois, usage faible, terminal récent ; 274 clients seulement, IC de 71 à 81 %).

- Les **5 segments les plus risqués sont tous à 11-12 mois d'ancienneté** : la fin d'engagement domine tous les autres critères.
- À ancienneté égale, un **usage faible** et un **terminal vieux** aggravent le risque.
- Les 10 segments à 55 % et plus regroupent 34,9 % des clients avec 59,8 % de churn ; les 5 segments sous 45 % regroupent 34,1 % des clients avec 36,8 %.

Ces règles fournissent un **benchmark simple** : le modèle devra faire mieux que cette segmentation pour justifier sa complexité (à comparer en E7-E8).

# Insights métier

Taux mesurés sur le train équilibré (moyenne 49,6 %) : ils comparent des groupes, ce ne sont pas des taux réels d'opérateur. Les liens sont des associations, pas des causes.

1. **La fin d'engagement est le premier facteur de départ.** À 11-12 mois d'ancienneté (14,1 % des clients), le churn atteint 63,5 %, contre 33,8 % entre 6 et 10 mois. C'est la période à cibler en priorité pour les offres de réengagement.
2. **L'âge du terminal confirme l'échéance.** Le churn saute de 42 % à 59 % entre 300 et 309 jours d'âge du terminal (56,5 % au-delà de 305 jours contre 39,5 % en deçà). Hypothèse : l'engagement de 12 mois se termine pendant la fenêtre d'observation. Proposer un renouvellement de terminal **avant 10 mois** est une piste d'action.
3. **Un terminal bon marché signale un client peu attaché.** À 9,99 $ ou 29,99 $ (26 % des clients), 57-58 % de churn, contre 41 % à 199,99 $.
4. **La baisse d'usage précède le départ.** Une chute de plus de 240 minutes : 55,4 % de churn, contre 45 % en cas de hausse. Les clients sans mesure de variation (718) churnent à 76 % et ceux sans aucun usage à 69 % : signal très fort mais peut-être déjà la **conséquence** du départ (D32, à tester en E8).
5. **Hypothèse : le contact avec le service client n'est pas un signal de départ.** Les clients qui appellent churnent un peu moins (46,9 % contre 51,7 %), et l'écart persiste à usage égal (2 à 5 points). L'écart est faible et plusieurs explications restent possibles : client engagé, offre de rétention faite pendant l'appel, problème résolu. Sans le motif ni l'issue des appels, on ne peut pas trancher.
6. **Les petits forfaits et les petits utilisateurs partent plus.** Forfait de moins de 30 $ : 53-57 % ; de 60 $ et plus : 43-44 %. Usage de moins de 49 min par mois : 58,9 %.
7. **La segmentation par règles isole un tiers de clients à risque.** 10 segments sur 21, soit 34,9 % des clients, concentrent 59,8 % de churn ; l'écart entre segments extrêmes atteint 47 points. C'est le benchmark que le modèle devra battre.
8. **Le signal de chaque variable reste faible.** La plus forte corrélation avec le churn est ρ = 0,13 (`eqpdays`), le plus fort V de Cramér 0,09 (`hnd_webcap`), et de nombreuses variables sont redondantes (ρ = 1,00 pour `totrev`/`adjrev`). La performance viendra de la **combinaison** des variables et de leurs interactions, pas d'un facteur unique : une AUC modérée est attendue.